### Preparation for JMeter test
- creation of query files
- generation of jmx files (ingestion, power test, concurrency)
- jdbc drivers install

In [ ]:
%run "./helpers/Helper Functions"

In [ ]:
# --- Create Widgets (text) ---
dbutils.widgets.removeAll()

dbutils.widgets.text("dbx_catalog", "", "Databricks Catalog")
dbutils.widgets.text("dbx_schema", "", "Databricks Schema")
dbutils.widgets.text("dbx_volume", "", "Volume name (for results)")
dbutils.widgets.text("dbx_warehouse_url", "", "Databricks Warehouse URL")
dbutils.widgets.text("dbx_pat_token", "", "Databricks PAT TOKEN")
dbutils.widgets.text("table_prefix", "", "Table Prefix")
dbutils.widgets.text("dbx_optimize_strategy", "zorder", "Optimize Strategy")
dbutils.widgets.text("dbx_optimize_strategy_slug", "", "Optimize Strategy Slug (auto-computed if blank)")

In [ ]:
import os
# --- Retrieve Values, Set Env Vars, and Build Dictionary ---

params = dbutils.widgets.getAll()

for k,v in params.items():
    os.environ[f"W_{k.upper()}"] = v
    
# --- Validation Check (all required) ---

if params["dbx_catalog"] == "":
    raise RuntimeError("Validation Error: Parameter 'dbx_catalog' is required but is empty.")

if params["dbx_schema"] == "":
    raise RuntimeError("Validation Error: Parameter 'dbx_schema' is required but is empty.")

if params["dbx_volume"] == "":
    raise RuntimeError("Validation Error: Parameter 'dbx_volume' is required but is empty.")

if params["dbx_warehouse_url"] == "":
    raise RuntimeError("Validation Error: Parameter 'dbx_warehouse_url' is required but is empty.")

if params["dbx_pat_token"] == "":
    raise RuntimeError("Validation Error: Parameter 'dbx_pat_token' is required but is empty.")

In [ ]:
import os, requests
from urllib.parse import urlparse

current_path = os.getcwd()
table_prefix = params.get("table_prefix", "").rstrip("_")
volume_base = f"/Volumes/{params['dbx_catalog']}/{params['dbx_schema']}/{params['dbx_volume']}"
conf_dir = f"{volume_base}/test_files/{table_prefix}" if table_prefix else f"{volume_base}/test_files"
os.makedirs(conf_dir, exist_ok=True)

raw_url = params["dbx_warehouse_url"]

# If the value is a warehouse size (not a JDBC URL), find or create the warehouse
dbx_strategy_raw = params.get("dbx_optimize_strategy", "zorder")
dbx_strategy_slug = params.get("dbx_optimize_strategy_slug", "").strip() or strategy_slug(dbx_strategy_raw)
if not raw_url.startswith("jdbc:"):
    host = spark.conf.get("spark.databricks.workspaceUrl")
    wh_id, raw_url, _ = find_or_create_databricks_warehouse(raw_url, params["scale"], dbx_strategy_slug, host, params["dbx_pat_token"])

# Restructure the user's warehouse URL for the Simba JDBC driver.
# User provides: jdbc:databricks://host:port/sql/1.0/warehouses/xxx
# Simba expects:  jdbc:databricks://host:port;httpPath=/sql/1.0/warehouses/xxx;...
parsed = urlparse(raw_url.replace("jdbc:databricks:", "https:"))
host_port = "{}:{}".format(parsed.hostname, parsed.port or 443)
http_path = parsed.path  # e.g. /sql/1.0/warehouses/xxx
dbx_jdbc_url = (
    "jdbc:databricks://{};httpPath={};"
    "UseNativeQuery=1;EnableArrow=0;"
    "ConnCatalog={};ConnSchema={};"
    "SocketTimeout=0"
).format(host_port, http_path, params["dbx_catalog"], params["dbx_schema"])

fill_params = {
    "JDBC_DRIVER": "/tpcds/db_driver.jar",
    "DBX_JDBC_URL": dbx_jdbc_url,
    "DBX_WAREHOUSE_TOKEN": params["dbx_pat_token"],
    "CURPATH": "/tmp/tpcds"  # runtime dir where 5a/5b/5c stage test_conf/ and JMeter writes results/,
}

# --- Ingestion JMX (Phase 1: Create Tables + Phase 2: Optimize) ---
template = get_jmx_template("DATABRICKS", "INGESTION")
jmx = fill_template(template, fill_params)

dbx_ingestion_path = f"{conf_dir}/databricks_ingestion.jmx"
if os.path.exists(dbx_ingestion_path):
    os.remove(dbx_ingestion_path)
with open(dbx_ingestion_path, "w") as f:
    f.write(jmx)

os.environ["JMX_DATABRICKS_INGESTION"] = dbx_ingestion_path

# --- Power Test JMX (Phase 3: SELECT queries) ---
template = get_jmx_template("DATABRICKS", "POWER_TEST")
jmx = fill_template(template, fill_params)

dbx_power_test_path = f"{conf_dir}/databricks_power_test.jmx"
if os.path.exists(dbx_power_test_path):
    os.remove(dbx_power_test_path)
with open(dbx_power_test_path, "w") as f:
    f.write(jmx)

os.environ["JMX_DATABRICKS_POWER_TEST"] = dbx_power_test_path

In [ ]:
import os
from urllib.parse import urlparse

current_path = os.getcwd()

raw_url = params["dbx_warehouse_url"]

# If the value is a warehouse size (not a JDBC URL), find or create the warehouse
dbx_strategy_raw = params.get("dbx_optimize_strategy", "zorder")
dbx_strategy_slug = params.get("dbx_optimize_strategy_slug", "").strip() or strategy_slug(dbx_strategy_raw)
if not raw_url.startswith("jdbc:"):
    host = spark.conf.get("spark.databricks.workspaceUrl")
    wh_id, raw_url, _ = find_or_create_databricks_warehouse(raw_url, params["scale"], dbx_strategy_slug, host, params["dbx_pat_token"])

parsed = urlparse(raw_url.replace("jdbc:databricks:", "https:"))
host_port = "{}:{}".format(parsed.hostname, parsed.port or 443)
http_path = parsed.path
dbx_jdbc_url = (
    "jdbc:databricks://{};httpPath={};"
    "UseNativeQuery=1;EnableArrow=0;"
    "ConnCatalog={};ConnSchema={};"
    "SocketTimeout=0"
).format(host_port, http_path, params["dbx_catalog"], params["dbx_schema"])

template = get_jmx_template("DATABRICKS", "CONCURRENCY")
jmx_concurrency = fill_template(template, {
    "JDBC_DRIVER": "/tpcds/db_driver.jar",
    "DBX_JDBC_URL": dbx_jdbc_url,
    "DBX_WAREHOUSE_TOKEN": params["dbx_pat_token"],
    "CURPATH": "/tmp/tpcds"  # runtime dir where 5a/5b/5c stage test_conf/ and JMeter writes results/,
})

dbx_jmx_path = f"{conf_dir}/databricks_concurrency.jmx"
if os.path.exists(dbx_jmx_path):
    os.remove(dbx_jmx_path)
with open(dbx_jmx_path, "w") as f:
    f.write(jmx_concurrency)

os.environ["JMX_DATABRICKS_CONCURRENCY"] = dbx_jmx_path

In [ ]:
import os

current_path = os.getcwd()

generate_concurrency_streams(
    perf_queries_tsv=f"{conf_dir}/tpcds-queries-perf-databricks.tsv",
    concurrency_json=f"{current_path}/resources/tpcds-concurrency-query-order.json",
    output_dir=f"{conf_dir}/tpcds_databricks_concurrency_streams",
)